# 🚀 SpaceGen AI — Free Cloud GPU Backend
**3D Gaussian Splatting on a Tesla T4 GPU · Zero install · Zero cost**

### How it works
1. **Runtime → Change runtime type → T4 GPU** (do this first!)
2. Run **Cell 1** to install everything (~2 min)
3. Run **Cell 2** to start the backend — copy the public URL into your frontend `.env.local`
4. Upload a video in the web app — it trains on the T4 and streams the 3D scene back!

In [ ]:
#@title Cell 1 — Install COLMAP + Original Gaussian Splatting + Cloudflare Tunnel
import os, subprocess, sys

os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['DISPLAY'] = ''

print('📦 [1/6] Installing system packages (COLMAP, FFmpeg)...')
!apt-get update -qq 2>&1 | tail -1
!apt-get install -y -qq colmap ffmpeg > /dev/null 2>&1
print('   ✅ COLMAP + FFmpeg installed')

print('📦 [2/6] Installing Python packages...')
!pip install -q plyfile tqdm pillow fastapi uvicorn python-multipart > /dev/null 2>&1
print('   ✅ FastAPI + PLY tools installed')

print('📦 [3/6] Cloning Gaussian Splatting repo...')
GS_DIR = '/content/gaussian-splatting'
if not os.path.exists(GS_DIR):
    !git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting {GS_DIR} 2>&1 | tail -2
else:
    print('   (already cloned)')
print('   ✅ Repo ready')

print('📦 [4/6] Installing Gaussian Splatting Python dependencies...')
# Install ALL dependencies the training script needs
!pip install -q lpips tensorboard > /dev/null 2>&1
# Install from the repo's own environment.yml / setup needs
!cd {GS_DIR} && pip install -q -e . 2>&1 | tail -2 || true
print('   ✅ Python deps installed (lpips, tensorboard, etc.)')

print('📦 [5/6] Compiling CUDA rasterizer + kNN (this takes ~60 s)...')
!pip install -q {GS_DIR}/submodules/diff-gaussian-rasterization 2>&1 | tail -1
!pip install -q {GS_DIR}/submodules/simple-knn 2>&1 | tail -1
print('   ✅ CUDA kernels compiled')

print('📦 [6/6] Installing Cloudflare Tunnel...')
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print('   ✅ cloudflared installed')

# Quick sanity checks
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print(f'\n🖥️  Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}  |  GPU: {gpu_name}')
!colmap -h 2>&1 | head -1

# Verify training script can import
print('\n🔍 Verifying train.py imports...')
verify = subprocess.run(
    ['python', '-c', 'import sys; sys.path.insert(0,\"/content/gaussian-splatting\"); from scene import Scene; from gaussian_renderer import render; print(\"OK\")'],
    capture_output=True, text=True
)
if verify.returncode == 0:
    print(f'   ✅ train.py imports verified: {verify.stdout.strip()}')
else:
    print(f'   ❌ Import error: {verify.stderr}')

print('\n✅ Everything installed! Run Cell 2 now.')

In [ ]:
#@title Cell 2 — Start FastAPI Backend + Cloudflare Public URL
import os, uuid, shutil, subprocess, threading, time, json, traceback
from pathlib import Path
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
import uvicorn

os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['DISPLAY'] = ''

GS_DIR = '/content/gaussian-splatting'
DATA_ROOT = Path('/content/spacegen_jobs')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
JOBS: dict = {}

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def run(cmd: list[str], cwd: str | None = None):
    """Run a shell command, stream output, raise on failure with full output."""
    env = os.environ.copy()
    env['QT_QPA_PLATFORM'] = 'offscreen'
    env['DISPLAY'] = ''
    print(f'  → {" ".join(cmd)}')
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env=env, cwd=cwd
    )
    output_lines = []
    for line in proc.stdout:
        stripped = line.rstrip()
        if stripped:
            output_lines.append(stripped)
            print(f'    {stripped}')
    proc.wait()
    if proc.returncode != 0:
        # Print last 30 lines for debugging
        tail = '\n'.join(output_lines[-30:])
        raise RuntimeError(f'Command exited {proc.returncode}\nLast output:\n{tail}')


def extract_frames(video: Path, out_dir: Path, target: int = 150):
    """Extract evenly-spaced frames using ffmpeg."""
    out_dir.mkdir(parents=True, exist_ok=True)
    probe = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
         '-count_packets', '-show_entries', 'stream=nb_read_packets',
         '-of', 'csv=p=0', str(video)],
        capture_output=True, text=True
    )
    total = int(probe.stdout.strip() or 0)
    if total <= 0:
        run(['ffmpeg', '-y', '-i', str(video), '-qscale:v', '2',
             '-vf', 'fps=2', str(out_dir / 'frame_%04d.jpg')])
        return
    step = max(1, total // target)
    run(['ffmpeg', '-y', '-i', str(video), '-qscale:v', '2',
         '-vf', f'select=not(mod(n\\,{step})),setpts=N/FRAME_RATE/TB',
         '-vsync', 'vfr', str(out_dir / 'frame_%04d.jpg')])


def run_colmap(image_dir: Path, workspace: Path):
    """Run COLMAP and undistort into the format expected by graphdeco 3DGS.

    The original trainer rejects OPENCV/FULL_OPENCV cameras and only
    accepts PINHOLE or SIMPLE_PINHOLE data."""
    db = workspace / 'database.db'
    sparse = workspace / 'sparse'
    sparse.mkdir(parents=True, exist_ok=True)

    run(['colmap', 'feature_extractor',
         '--database_path', str(db),
         '--image_path', str(image_dir),
         '--ImageReader.single_camera', '1',
         '--ImageReader.camera_model', 'OPENCV',
         '--SiftExtraction.use_gpu', '0'])

    run(['colmap', 'exhaustive_matcher',
         '--database_path', str(db),
         '--SiftMatching.use_gpu', '0'])

    run(['colmap', 'mapper',
         '--database_path', str(db),
         '--image_path', str(image_dir),
         '--output_path', str(sparse)])

    # Verify sparse/0 exists (the training script requires this)
    if not (sparse / '0').exists():
        # Use the first model directory that exists
        model_dirs = sorted([d for d in sparse.iterdir() if d.is_dir()])
        if model_dirs:
            print(f'  ℹ️  Renaming {model_dirs[0].name}/ → 0/')
            model_dirs[0].rename(sparse / '0')
        else:
            raise RuntimeError('COLMAP produced no sparse model')

    # The graphdeco trainer rejects OPENCV/FULL_OPENCV cameras. Convert
    # the reconstruction and replace the dataset with undistorted data.
    undistorted = workspace / 'undistorted'
    if undistorted.exists():
        shutil.rmtree(undistorted)
    run(['colmap', 'image_undistorter',
         '--image_path', str(image_dir),
         '--input_path', str(sparse / '0'),
         '--output_path', str(undistorted),
         '--output_type', 'COLMAP'])

    undistorted_images = undistorted / 'images'
    undistorted_sparse = undistorted / 'sparse'
    undistorted_model = undistorted_sparse / '0' if (undistorted_sparse / '0').exists() else undistorted_sparse
    if not undistorted_images.exists() or not (undistorted_model / 'cameras.bin').exists():
        raise RuntimeError('COLMAP undistortion did not create images/ and sparse/cameras.bin')
    shutil.rmtree(image_dir)
    shutil.move(str(undistorted_images), str(image_dir))
    shutil.rmtree(sparse)
    sparse.mkdir(parents=True, exist_ok=True)
    shutil.move(str(undistorted_model), str(sparse / '0'))
    shutil.rmtree(undistorted, ignore_errors=True)

    n_images = len(list((sparse / '0').glob('*.bin')))
    print(f'  ✅ COLMAP undistorted dataset ready: PINHOLE-compatible sparse/0/ ({n_images} files)')


def verify_3dgs_dataset(dataset_dir: Path):
    """Validate the dataset before starting GPU training."""
    images = dataset_dir / 'images'
    sparse0 = dataset_dir / 'sparse' / '0'
    required = [images, sparse0 / 'cameras.bin', sparse0 / 'images.bin', sparse0 / 'points3D.bin']
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise RuntimeError('Dataset is incomplete after COLMAP: ' + ', '.join(missing))
    image_count = sum(len(list(images.glob(pattern))) for pattern in ('*.jpg', '*.jpeg', '*.png'))
    if image_count < 3:
        raise RuntimeError(f'Only {image_count} undistorted images are available; capture at least 3 overlapping views.')
    converted = sparse0.parent / 'validation_txt'
    if converted.exists():
        shutil.rmtree(converted)
    check = subprocess.run(['colmap', 'model_converter', '--input_path', str(sparse0), '--output_path', str(converted), '--output_type', 'TXT'], capture_output=True, text=True)
    if check.returncode != 0:
        raise RuntimeError('Could not inspect the COLMAP camera model: ' + check.stderr[-500:])
    camera_file = converted / 'cameras.txt'
    camera_models = []
    if camera_file.exists():
        for line in camera_file.read_text().splitlines():
            if line and not line.startswith('#'):
                fields = line.split()
                if len(fields) >= 2:
                    camera_models.append(fields[1])
    shutil.rmtree(converted, ignore_errors=True)
    unsupported = [m for m in camera_models if m not in {'PINHOLE', 'SIMPLE_PINHOLE'}]
    if unsupported:
        raise RuntimeError(f'COLMAP camera model still unsupported after undistortion: {sorted(set(unsupported))}')
    print(f'  ✅ Dataset validation passed: {image_count} images, cameras={sorted(set(camera_models))}')


def train_gaussian_splatting(dataset_dir: Path, model_dir: Path, iterations: int = 7000):
    """
    Run the original 3D Gaussian Splatting trainer.
    Uses 7000 iterations (good quality, fits in T4 time budget).
    dataset_dir must contain images/ and sparse/0/.
    """
    model_dir.mkdir(parents=True, exist_ok=True)
    run([
        'python', 'train.py',
        '-s', str(dataset_dir),
        '-m', str(model_dir),
        '--iterations', str(iterations),
        '--save_iterations', str(iterations),
        '--quiet',
    ], cwd=GS_DIR)


# ---------------------------------------------------------------------------
# Background worker
# ---------------------------------------------------------------------------

def _worker(job_id: str):
    job = JOBS[job_id]
    job_dir = DATA_ROOT / job_id
    video_path = job_dir / 'capture.mp4'
    dataset = job_dir / 'dataset'
    image_dir = dataset / 'images'
    model_dir = job_dir / 'model'

    try:
        # STEP 1: Extract frames
        job.update(step='Extracting video frames', progress=10)
        print(f'\n🎬 [{job_id}] Step 1/3 — Extracting frames...')
        extract_frames(video_path, image_dir)
        n_frames = len(list(image_dir.glob('*.jpg')))
        print(f'   Extracted {n_frames} frames')
        job.update(log=f'Extracted {n_frames} frames')

        # STEP 2: COLMAP SfM
        job.update(step='Estimating camera poses with COLMAP', progress=30)
        print(f'📐 [{job_id}] Step 2/3 — Running COLMAP SfM (CPU SIFT)...')
        run_colmap(image_dir, dataset)
        verify_3dgs_dataset(dataset)
        job.update(log='COLMAP SfM finished')

        # STEP 3: Train Gaussian Splatting
        job.update(step='Training 3D Gaussians on Tesla T4 GPU', progress=55)
        print(f'🧠 [{job_id}] Step 3/3 — Training Gaussian Splatting (7000 iters)...')
        train_gaussian_splatting(dataset, model_dir, iterations=7000)

        # Find the output PLY
        ply_candidates = sorted(model_dir.rglob('point_cloud.ply'), key=lambda p: p.stat().st_mtime, reverse=True)
        if not ply_candidates:
            # Also check for the iteration-specific directory
            ply_candidates = sorted(model_dir.rglob('*.ply'), key=lambda p: p.stat().st_mtime, reverse=True)
        if not ply_candidates:
            raise FileNotFoundError(f'No PLY found. Contents of model dir: {list(model_dir.rglob("*"))}')

        ply_path = ply_candidates[0]
        ply_size_mb = ply_path.stat().st_size / (1024 * 1024)
        print(f'   ✅ PLY file: {ply_path} ({ply_size_mb:.1f} MB)')

        job.update(
            status='completed', progress=100,
            step='Scene ready',
            log=f'Done! PLY {ply_size_mb:.1f} MB',
            ply_path=str(ply_path),
        )
        print(f'\n🎉 JOB {job_id} COMPLETED!')

    except Exception as e:
        error_msg = str(e)
        print(f'\n❌ JOB {job_id} FAILED: {error_msg}')
        traceback.print_exc()
        job.update(status='failed', log=error_msg[:500])


# ---------------------------------------------------------------------------
# FastAPI app
# ---------------------------------------------------------------------------

app = FastAPI(title='SpaceGen Cloud Backend')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'], allow_credentials=True,
    allow_methods=['*'], allow_headers=['*'],
)


@app.get('/')
async def root():
    import torch
    return {
        'status': 'online',
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'service': 'SpaceGen Cloud Backend (Original 3DGS)',
    }


@app.post('/api/reconstruction/jobs', status_code=202)
async def create_job(video: UploadFile = File(...)):
    job_id = uuid.uuid4().hex[:12]
    job_dir = DATA_ROOT / job_id
    job_dir.mkdir(parents=True, exist_ok=True)
    video_path = job_dir / 'capture.mp4'
    with video_path.open('wb') as f:
        shutil.copyfileobj(video.file, f)
    JOBS[job_id] = {
        'job_id': job_id,
        'status': 'running',
        'progress': 5,
        'step': 'Video received, starting pipeline…',
        'log': '',
        'splat_url': f'/api/reconstruction/jobs/{job_id}/model',
    }
    threading.Thread(target=_worker, args=(job_id,), daemon=True).start()
    return JOBS[job_id]


def _resolve_ply(job_id: str) -> Path | None:
    if job_id in JOBS and 'ply_path' in JOBS[job_id]:
        p = Path(JOBS[job_id]['ply_path'])
        if p.exists():
            return p
    candidates = sorted(
        (DATA_ROOT / job_id).rglob('point_cloud.ply'),
        key=lambda p: p.stat().st_mtime, reverse=True
    ) if (DATA_ROOT / job_id).exists() else []
    return candidates[0] if candidates else None


@app.get('/api/reconstruction/jobs/{job_id}')
async def get_job(job_id: str):
    if job_id in JOBS:
        return JOBS[job_id]
    ply = _resolve_ply(job_id)
    if ply:
        return {
            'job_id': job_id, 'status': 'completed', 'progress': 100,
            'step': 'Scene ready', 'splat_url': f'/api/reconstruction/jobs/{job_id}/model',
        }
    raise HTTPException(404, 'Job not found')


@app.get('/api/reconstruction/jobs/latest')
async def get_latest_job():
    if JOBS:
        return list(JOBS.values())[-1]
    if DATA_ROOT.exists():
        dirs = sorted(
            [d for d in DATA_ROOT.iterdir() if d.is_dir()],
            key=lambda d: d.stat().st_mtime, reverse=True
        )
        for d in dirs:
            ply = _resolve_ply(d.name)
            if ply:
                return {
                    'job_id': d.name, 'status': 'completed', 'progress': 100,
                    'step': 'Scene ready',
                    'splat_url': f'/api/reconstruction/jobs/{d.name}/model',
                }
    raise HTTPException(404, 'No jobs yet')


@app.get('/api/reconstruction/jobs/{job_id}/model')
async def get_model(job_id: str):
    ply = _resolve_ply(job_id)
    if ply:
        return FileResponse(
            path=ply, media_type='application/octet-stream',
            filename='point_cloud.ply',
            headers={'Access-Control-Allow-Origin': '*'},
        )
    raise HTTPException(404, 'Model not found')


# ---------------------------------------------------------------------------
# Cloudflare Tunnel + Uvicorn
# ---------------------------------------------------------------------------

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

cloud_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    if 'trycloudflare.com' in line:
        for part in line.split():
            if 'https://' in part and 'trycloudflare.com' in part:
                cloud_url = part.strip().rstrip('.')
                break
        if cloud_url:
            break
    time.sleep(0.3)

print()
print('═' * 70)
print('  🚀  YOUR FREE CLOUD BACKEND URL')
print(f'  {cloud_url or "(still starting…)"}')
print()
print('  👉  Paste this in your frontend/.env.local:')
print(f'  NEXT_PUBLIC_API_URL={cloud_url}')
print('═' * 70)
print()

config = uvicorn.Config(app=app, host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)
await server.serve()
